In [1]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF file
file_path = "C:\\Users\\Técnico em IA\\Documents\\Arthur Steckelberg Técnico IA\\UC15 - NLP\\senac_ia_uc15_nlp_projeto\\senac_ia_uc15_nlp_project\\data\\marcelo\\sindilojas_2025_2026.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

38


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Split the documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

229


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

# Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2") # sentence-transformers/all-mpnet-base-v2

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Embed the first two chunks
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[-0.19850429892539978, 0.0832478255033493, -0.014325892552733421, 0.1196397989988327, 0.14746317267417908, -0.05140233412384987, 0.016535399481654167, -0.023427538573741913, -0.01966334693133831, 0.044065505266189575]


In [12]:
from langchain_chroma import Chroma

# Create a vector store
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [13]:
ids = vector_store.add_documents(documents=all_splits)

In [14]:
results = vector_store.similarity_search(
    "Quais obrigações a empresa deve cumprir, em caso de parcelamento da rescisão?"
)

print(results[0])

page_content='de São Paulo, acompanhado da comprovação dos descontos e do efetivo recolh imento dos 
valores reclamados, até o encerramento da instrução processual. Em caso de condenação da 
empresa na devolução desses valores o sindicato da categoria profi ssional deverá ressarci-la, 
na cota correspondente ao crédito do sindicato, no prazo máximo de 30 (trinta) dias, 
contados do trânsito em julgado da sentença condenatória ou da homol ogação do acordo' metadata={'page_label': '7', 'producer': 'Microsoft® Word para Microsoft 365 - by Lacuna Software PKI SDK', 'author': 'Vivianne Morais', 'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'page': 6, 'creationdate': '2025-11-24T18:02:52-03:00', 'moddate': '2025-11-24T21:54:55+00:00', 'creator': 'Microsoft® Word para Microsoft 365', 'total_pages': 38, 'start_index': 753, 'source': 'C:\\Users\\Técnico em IA\\Documents\\Arthur Steckelberg Técnico IA\\UC15 - NLP\\senac_ia_uc15_nlp_projeto\\senac_ia_uc15_nlp_project\\data\\marcelo\\sind